In [47]:
import matplotlib
import matplotlib.pyplot as plt
import backtrader as bt
import pandas as pd

In [48]:
class MyBuySell(bt.observers.BuySell):
    plotlines = dict(
        buy=dict(marker='^', markersize=8.0, color='blue', fillstyle='full'),
        sell=dict(marker='v', markersize=8.0, color='red', fillstyle='full')
    )

In [49]:
class SmaStrategy(bt.Strategy):
    params = (('ma_period', 20), )

    def __init__(self):
        
        #종가(close) 추적
        self.data_close = self.datas[0].close

         #주문(order)/주가(price)/수수료(comm) 추적
        self.order = None
        self.price = None
        self.comm = None

        #분석기간
        print("단순 이동 평균(SMA) 분석기간 =", self.params.ma_period)

        #단순 이동 평균(SMA) 추적
        self.sma = bt.ind.SMA(self.datas[0], period=self.params.ma_period)

    def log(self, txt):
        '''Logging function'''
        dt = self.datas[0].datetime.date(0).isoformat()
        print(f'{dt}, {txt}')

    # 주문(Order)의 상태가 바뀌었을 때 호출하는 함수
    def notify_order(self, order):
        if order.status in [order.Submitted, order.Accepted]: #Submitted(주문제출), Accepted(주문접수)는 실제로 거래가 이루어진 것이 아니기에 return
            return

        if order.status in [order.Completed]:                 #Completed(거래체결)
            if order.isbuy():
                self.log(f'매수 주문 완료 --- 주문가(Price): {order.executed.price:.2f}, 거래가(Cost): {order.executed.value:.2f}, 수수료(Commission): {order.executed.comm:.2f}')
                self.price = order.executed.price
                self.comm = order.executed.comm
            else:
                self.log(f'매도 주문 완료 --- 주문가(Price): {order.executed.price:.2f}, 거래가(Cost): {order.executed.value:.2f}, 수수료(Commission): {order.executed.comm:.2f}')

        elif order.status in [order.Canceled, order.Margin, order.Rejected]:  #Canceled(주문취소), Margin(자금부족), Rejected(주문거부)
            self.log(f'주문 실패 확인 --- 사유: {order.getstatusname()}')

        self.order = None
    
    #거래(Trade)가 끝났을 때 그 거래의 손익을 알려주는 함수
    def notify_trade(self, trade):
        if not trade.isclosed:
            return

        self.log(f'거래 결과 확인 --- 거래수익(Gross): {trade.pnl:.2f}, 순수익(Net): {trade.pnlcomm:.2f}')

    #매매 판단 및 주문을 수행
    def next(self):

        if self.order:
            return

        if not self.position:

            if self.data_close[0] > self.sma[0]:
                self.log(f'매수 주문 생성 --- 매수가(Price): {self.data_close[0]:.2f}')
                self.order = self.buy()
        else:
    
            if self.data_close[0] < self.sma[0]:
                self.log(f'매도 주문 생성 --- 매도가(Price): {self.data_close[0]:.2f}')
                self.order = self.sell()

In [50]:
class FinalValue(bt.Analyzer):

    def stop(self):
        self.rets['final_value'] = self.strategy.broker.getvalue()

#### 백테스트용 데이터 불러오기(애플 2018년도)

In [51]:
dir_nm = "dailyStock"
target = "AAPL"
file_path = f"{dir_nm}/{target}.csv"

aapl_df = pd.read_csv(file_path, encoding="utf-8")
aapl_df['Date'] = pd.to_datetime(aapl_df['Date'])
aapl_df = aapl_df.set_index("Date")

aapl_df = aapl_df.loc["2018-01-01":"2018-12-31"]

data = bt.feeds.PandasData(dataname=aapl_df)

### 백테스트 설정

In [52]:
cerebro = bt.Cerebro(stdstats = False)

cerebro.adddata(data)
cerebro.optstrategy(SmaStrategy, ma_period=range(10, 31))
cerebro.addanalyzer(FinalValue,_name='final_value')
cerebro.broker.setcash(1000.0)

### 백테스트 실행

In [53]:
results = cerebro.run(maxcpus=1, optreturn=False)

단순 이동 평균(SMA) 분석기간 = 10
2018-01-16, 매수 주문 생성 --- 매수가(Price): 44.05
2018-01-17, 매수 주문 완료 --- 주문가(Price): 44.04, 거래가(Cost): 44.04, 수수료(Commission): 0.00
2018-01-24, 매도 주문 생성 --- 매도가(Price): 43.56
2018-01-25, 매도 주문 완료 --- 주문가(Price): 43.63, 거래가(Cost): 44.04, 수수료(Commission): 0.00
2018-01-25, 거래 결과 확인 --- 거래수익(Gross): -0.41, 순수익(Net): -0.41
2018-02-12, 매수 주문 생성 --- 매수가(Price): 40.68
2018-02-13, 매수 주문 완료 --- 주문가(Price): 40.49, 거래가(Cost): 40.49, 수수료(Commission): 0.00
2018-03-07, 매도 주문 생성 --- 매도가(Price): 43.76
2018-03-08, 매도 주문 완료 --- 주문가(Price): 43.87, 거래가(Cost): 40.49, 수수료(Commission): 0.00
2018-03-08, 거래 결과 확인 --- 거래수익(Gross): 3.38, 순수익(Net): 3.38
2018-03-08, 매수 주문 생성 --- 매수가(Price): 44.24
2018-03-09, 매수 주문 완료 --- 주문가(Price): 44.49, 거래가(Cost): 44.49, 수수료(Commission): 0.00
2018-03-16, 매도 주문 생성 --- 매도가(Price): 44.51
2018-03-19, 매도 주문 완료 --- 주문가(Price): 44.33, 거래가(Cost): 44.49, 수수료(Commission): 0.00
2018-03-19, 거래 결과 확인 --- 거래수익(Gross): -0.16, 순수익(Net): -0.16
2018-04-04, 매수 주문 생성 --- 매수가(Pric

### 최적화 결과 분석

In [56]:
for result in results:
    strat = result[0]

    final_value = strat.analyzers.final_value.get_analysis()['final_value']

    print(f"분석기간 {strat.params.ma_period:2d}일 : 포트폴리오 종료가({final_value:.2f})")

분석기간 10일 : 포트폴리오 종료가(1001.07)
분석기간 11일 : 포트폴리오 종료가(1000.78)
분석기간 12일 : 포트폴리오 종료가(1000.27)
분석기간 13일 : 포트폴리오 종료가(997.73)
분석기간 14일 : 포트폴리오 종료가(993.86)
분석기간 15일 : 포트폴리오 종료가(993.27)
분석기간 16일 : 포트폴리오 종료가(994.55)
분석기간 17일 : 포트폴리오 종료가(997.63)
분석기간 18일 : 포트폴리오 종료가(998.34)
분석기간 19일 : 포트폴리오 종료가(1000.99)
분석기간 20일 : 포트폴리오 종료가(1002.61)
분석기간 21일 : 포트폴리오 종료가(1002.48)
분석기간 22일 : 포트폴리오 종료가(1005.05)
분석기간 23일 : 포트폴리오 종료가(1004.40)
분석기간 24일 : 포트폴리오 종료가(1004.40)
분석기간 25일 : 포트폴리오 종료가(1004.28)
분석기간 26일 : 포트폴리오 종료가(1002.03)
분석기간 27일 : 포트폴리오 종료가(1001.73)
분석기간 28일 : 포트폴리오 종료가(1002.67)
분석기간 29일 : 포트폴리오 종료가(1003.53)
분석기간 30일 : 포트폴리오 종료가(1003.66)
